In [ ]:
simfin_api_key = os.getenv("SIMFIN_API_KEY")
# Set your API-key for downloading data. This key gets the free data.
sf.set_api_key(simfin_api_key)
# Set the local directory where data-files are stored.
# The directory will be created if it does not already exist.
sf.set_data_dir('~/simfin_data/')

In [ ]:
# We are interested in the US stock-market.
market = 'us'

# Add this date-offset to the fundamental data such as
# Income Statements etc., because the REPORT_DATE is not
# when it was actually made available to the public,
# which can be 1, 2 or even 3 months after the Report Date.
offset = pd.DateOffset(days=30)

# Refresh the fundamental datasets (Income Statements etc.)
# every 30 days.
refresh_days = 30

# Refresh the dataset with shareprices every 10 days.
refresh_days_shareprices = 7

In [ ]:
%%time
hub = sf.StockHub(market=market, offset=offset,
                  refresh_days=refresh_days,
                  refresh_days_shareprices=refresh_days_shareprices)

In [ ]:
REPORT_DATE, PUBLISH_DATE, RESTATED_DATE = 'Report Date', 'Publish Date', 'Restated Date'

In [ ]:
df_fin_signals = hub.fin_signals(variant='daily')

In [ ]:
%%time
df_growth_signals = hub.growth_signals(variant='daily')

In [ ]:
%%time
df_val_signals = hub.val_signals(variant='daily')

In [ ]:
%%time
# Combine the DataFrames.
dfs = [df_fin_signals, df_growth_signals, df_val_signals]
df_signals = pd.concat(dfs, axis=1)

In [ ]:
df_signals.head()

In [ ]:
df_signals.shape

In [ ]:
df_signals.columns

In [ ]:
df.index

In [ ]:
stock_symbols = ['AAPL', 'MSFT', 'AMZN', 'TSLA']

In [ ]:
df_signals_to_calc = df_signals.loc[stock_symbols]

In [ ]:
df_signals_to_calc.shape

In [ ]:
# Remove all rows with only NaN values.
df_signals_to_calc = df_signals_to_calc.dropna(how='all').reset_index(drop=False)

In [ ]:
df_signals_to_calc.head()

In [ ]:
df_signals_to_calc['Date'].min(), df_signals_to_calc['Date'].max()

In [ ]:
tickers = df_signals.groupby(level='Ticker').agg({'Current Ratio': 'count'})
tickers = tickers.rename(columns={'Current Ratio': 'CountRows'})

In [ ]:
tickers.shape

In [ ]:
tickers.head()

In [ ]:
tickers[tickers.index == 'AAPL']

In [ ]:
from Utils import merge_df_to_pg, get_conn_pg_engine
import SimfinLib as sfl

In [ ]:
# Create SQLAlchemy engine
engine = get_conn_pg_engine(False)
table_name = 'simfin_stats'
schema_name = 'finance'
merge_key_list = ['report_date', 'ticker']

In [ ]:
for ticker in tickers.index:
    print(ticker)
    if '_delisted' not in ticker:        
        df_ticker = df_signals.xs(ticker, level='Ticker')
        df_ticker['ticker'] = ticker
        df_ticker = sfl.simfin_rename_columns('stats', df_ticker)
        df_ticker = df_ticker.reset_index("report_date")
        merge_df_to_pg(df_ticker, engine, table_name, schema_name, merge_key_list)

In [ ]:
ticker = 'AAPL'
df_ticker = df_signals.xs(ticker, level='Ticker')
df_ticker['ticker'] = ticker
df_ticker = sfl.simfin_rename_columns('stats', df_ticker)
df_ticker = df_ticker.reset_index("report_date")
merge_df_to_pg(df_ticker, engine, table_name, schema_name, merge_key_list)

In [ ]:
df_ticker.head(10)

In [ ]:
import SimfinLib as sfl

In [ ]:
sfl.load_simfin()

In [ ]:
sfl.simfin_clear_cache()